In [ ]:
import math_toolkit
math_toolkit.notebook.activate()

# DoIt

Evaluate pending SymPy operations without breaking a `>>` transformation pipeline.

`DoIt(expr)`  
`expr >> DoIt`  
`expr >> DoIt(deep=False)`

In [ ]:
area = Integral(x, (x, 0, 1))

area @Eq@ (area >> DoIt)

## Core behavior

`DoIt` is the pipe-aware form of SymPy's `.doit()` method. It asks an expression to carry out operations that were deliberately left unevaluated, such as `Integral`, `Sum`, `Product`, `Limit`, and `Derivative`.

The operator does not invent a new evaluation rule. `DoIt(expr)` returns the same result as `expr.doit()`, and `expr >> DoIt` lets the same operation sit naturally among other pipeline stages.

In [ ]:
finite_sum = Sum(k**2, (k, 1, n))

DoIt(finite_sum) @Eq@ (finite_sum >> DoIt)

## Common patterns

**Evaluate after building readable notation.** Keep a formula in unevaluated mathematical form while it is being assembled, then call `DoIt` at the point where the value or closed form is wanted.

In [ ]:
power_sum = Sum(k**p, (k, 1, n))
squares_to_five = power_sum >> Subs({p: 2, n: 5}) >> DoIt

power_sum @Eq@ squares_to_five

**Finish pending derivatives created by symbolic rewrites.** Some transformations expose an unevaluated derivative. `DoIt` is the explicit final step that asks SymPy to run it.

In [ ]:
pending_slope = Derivative(sin(x)**2, x)

pending_slope @Eq@ (pending_slope >> DoIt >> Expand)

## Options and Details

`DoIt` forwards keyword hints to SymPy's `.doit(**hints)` method. The most common hint is `deep=False`, which asks SymPy to evaluate only the outer operation instead of recursively evaluating pending operations inside it.

Because `DoIt` is a `PipeOp`, a call with only hints creates a configured pipeline stage. That means `DoIt(deep=False)` can be prepared before the expression arrives from the left side of `>>`.

In [ ]:
nested = Integral(Integral(x, x), x)

md("""The default evaluates recursively, while `deep=False` leaves the inner operation pending.

- Default: {{ nested >> DoIt }}
- Shallow: {{ nested >> DoIt(deep=False) }}
""")

**Direct calls remain ordinary calls.** Use `DoIt(expr, deep=False)` when that reads better than a pipeline. Use `expr >> DoIt(deep=False)` when `DoIt` is one stage in a longer transformation.

In [ ]:
direct = DoIt(nested, deep=False)
piped = nested >> DoIt(deep=False)

direct @Eq@ piped

## Semantics

`DoIt` is explicit evaluation of pending operations, not general cleanup. If an expression has no pending `.doit()` work, the result may be unchanged. Use neighboring transforms such as `Simplify`, `Expand`, `Subs`, `Evalf`, and `Diff` for those separate operations.

The original expression is not mutated. SymPy returns a new expression or the original expression according to the usual immutability rules.

In [ ]:
identity = sin(x)**2 + cos(x)**2

md("""`DoIt` leaves ordinary algebra alone; `Simplify` handles the identity.

- DoIt: {{ identity >> DoIt }}
- Simplify: {{ identity >> Simplify }}
""")

## Examples and Applications

### Example: evaluate a limit after substitution

Build the limiting expression once, specialize a parameter later, and then evaluate the pending limit.

In [ ]:
scaled_limit = Limit((sin(a*x) / x), x, 0)
specialized_limit = scaled_limit >> Subs({a: 3}) >> DoIt

scaled_limit @Eq@ specialized_limit

### Example: combine exact evaluation with later numeric display

`DoIt` can produce the exact symbolic value first. If you want a decimal display, add `Evalf` as a separate stage so the symbolic and numeric decisions stay visible.

In [ ]:
geometric_area = Integral(exp(-x), (x, 0, 2))
decimal_area = geometric_area >> DoIt >> Evalf(8)

geometric_area @Eq@ decimal_area

## Pitfalls

**Do not use `DoIt` as a substitute for simplification.** It only calls the expression's pending-operation evaluator. Algebraic forms that need factoring, expansion, collection, or simplification should use the corresponding transform.

**Do not expect every pending operation to have a closed form.** SymPy may return an unevaluated object when it cannot evaluate an integral, sum, product, limit, or derivative under the available assumptions.

**Use the uppercase spelling.** `DoIt` is the notebook pipe operator exported by `math_toolkit`. The lowercase `.doit()` remains the ordinary SymPy method on expressions.

## See also

[PipeOps](PipeOps.ipynb)  
[ExprTransform](ExprTransform.ipynb)  
[rewrite](rewrite.ipynb)  
[subs](subs.ipynb)  
[Composing structured expressions](../tutorials/composing_expressions.ipynb)  
[Polynomial interpolation and unstable coefficients](../worksheets/polynomial_interpolation_instability.ipynb)